# 📘 Notebook 6: Probability Spaces, Events & the Kolmogorov Axioms

**Week 2 — Calculus, Optimization & Probability Theory**

**Difficulty:** ⭐⭐ (Beginner-Friendly)

---

## 🏠 Why Does This Matter?

Your house price model is never 100% certain. The real world is noisy:
- Two identical houses might sell at different prices (market timing, negotiation)
- Your training data is a **random sample** from all possible houses
- Your model should output a **confidence**, not just a point estimate

To reason rigorously about this uncertainty, we need **probability theory**.

And not "intuitive" probability — rigorous probability that won't break down in edge cases.  
That rigorous foundation was built by **Andrey Kolmogorov** in 1933 with just 3 axioms.

---

## 📚 Prerequisites
- Python (sets, lists, numpy)
- No prior probability knowledge required!

---

## Part 1: The Vocabulary of Probability

### Plain English First

Probability has precise vocabulary. Let's learn it through the house price context.

| Term | Technical Name | House Price Example |
|------|---------------|--------------------|
| A process with uncertain outcome | **Experiment** | Randomly picking a house from your dataset |
| The set of ALL possible outcomes | **Sample space Ω** | All houses that could possibly exist |
| One specific outcome | **Elementary event** | "This specific house sold for \$425,000" |
| A set of outcomes we care about | **Event A** | "House sold for more than \$400,000" |
| A number between 0 and 1 | **Probability P(A)** | "30% of houses sell above \$400,000" |

### Key Examples

- **Ω** = {all possible house prices from $50k to $5M}
- **Event A** = {price > $400k}  ← a subset of Ω
- **Event B** = {price < $200k} ← another subset
- **A ∩ B** = {price > $400k AND < $200k} = ∅ (impossible — disjoint!)
- **A ∪ B** = {price > $400k OR < $200k} ← all extreme prices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ─────────────────────────────────────────────────────────────────
# Simulate a housing market dataset
# We'll use this throughout the probability notebooks
# ─────────────────────────────────────────────────────────────────

np.random.seed(42)
N_HOUSES = 100_000   # simulate 100,000 house sales

# Generate house prices: log-normal distribution (prices are always positive
# and right-skewed, which matches real estate markets)
log_prices = np.random.normal(loc=np.log(300_000), scale=0.5, size=N_HOUSES)
prices     = np.exp(log_prices)   # convert back from log scale → prices in dollars

# Generate number of bedrooms (1 to 6, integer)
bedrooms   = np.random.choice([1, 2, 3, 4, 5, 6],
                               size=N_HOUSES,
                               p=[0.05, 0.25, 0.40, 0.20, 0.08, 0.02])

# ─── Define events as boolean arrays ──────────────────────────────
# Each event is a True/False array of length N_HOUSES
event_A  = prices > 400_000    # A: expensive house (>400k)
event_B  = prices < 200_000    # B: cheap house (<200k)
event_C  = bedrooms >= 4       # C: large house (4+ bedrooms)

def P(event):
    """Compute empirical probability: fraction of houses where event is True."""
    return np.mean(event)   # mean of boolean array = fraction of True values

print("House Market: 100,000 simulated house sales")
print(f"  Sample price range: ${prices.min():.0f} to ${prices.max():.0f}")
print(f"  Median price:       ${np.median(prices):.0f}")
print()
print("Basic probabilities:")
print(f"  P(A) = P(price > $400k)   = {P(event_A):.4f}  ({P(event_A)*100:.1f}% of houses)")
print(f"  P(B) = P(price < $200k)   = {P(event_B):.4f}  ({P(event_B)*100:.1f}% of houses)")
print(f"  P(C) = P(bedrooms ≥ 4)   = {P(event_C):.4f}  ({P(event_C)*100:.1f}% of houses)")
print(f"  P(Ω) = P(any house)       = {P(np.ones(N_HOUSES, dtype=bool)):.4f}  (= 1 always!)")
print(f"  P(∅) = P(impossible event)= {P(np.zeros(N_HOUSES, dtype=bool)):.4f}  (= 0 always!)")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Visualize the sample space and events as a histogram
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: price histogram with events highlighted ─────────────────
bins = np.linspace(0, 1_500_000, 100)

axes[0].hist(prices, bins=bins, color='steelblue', alpha=0.7,
             density=True, label='All houses (Ω)')
# Shade event A (>400k) in red
axes[0].hist(prices[event_A], bins=bins, color='red', alpha=0.6,
             density=True, label=f'Event A: price>$400k  P={P(event_A):.3f}')
# Shade event B (<200k) in green
axes[0].hist(prices[event_B], bins=bins, color='green', alpha=0.6,
             density=True, label=f'Event B: price<$200k  P={P(event_B):.3f}')

axes[0].set_xlabel('House Price ($)', fontsize=11)
axes[0].set_ylabel('Probability Density', fontsize=11)
axes[0].set_title('Sample Space Ω and Events as Regions of the Histogram', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 1_500_000)

# ── Right: Venn diagram showing A, B, C relationships ─────────────
axes[1].set_xlim(0, 4); axes[1].set_ylim(0, 3)
# Draw Ω as a rectangle
rect = plt.Rectangle((0.1, 0.1), 3.8, 2.8, fill=False, edgecolor='black', linewidth=3)
axes[1].add_patch(rect)
axes[1].text(3.5, 2.7, 'Ω\n(all houses)', fontsize=10, ha='right')

# Draw circles for events
from matplotlib.patches import Circle
circA = Circle((1.2, 1.5), 0.85, alpha=0.35, color='red',   label=f'A: price>$400k')
circB = Circle((2.1, 1.5), 0.85, alpha=0.35, color='green', label=f'B: price<$200k')
circC = Circle((2.8, 1.5), 0.85, alpha=0.35, color='blue',  label=f'C: beds≥4')
axes[1].add_patch(circA); axes[1].add_patch(circB); axes[1].add_patch(circC)

axes[1].text(0.8, 1.5, f'A\nP={P(event_A):.2f}', ha='center', fontsize=10, fontweight='bold')
axes[1].text(2.1, 1.5, f'B\nP={P(event_B):.2f}', ha='center', fontsize=10, fontweight='bold')
axes[1].text(3.0, 1.5, f'C\nP={P(event_C):.2f}', ha='center', fontsize=10, fontweight='bold')

# Note: A and B don't overlap (price can't be both >400k AND <200k)
axes[1].text(1.2, 0.35, 'A ∩ B = ∅\n(impossible!)', ha='center', fontsize=9, color='darkred')

axes[1].axis('off')
axes[1].set_title('Events as Subsets of Sample Space Ω\n(circles show which houses satisfy each condition)', fontsize=11)
axes[1].legend(loc='upper left', fontsize=9)

plt.suptitle('Probability: Events as Subsets of the Sample Space', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 2: The Kolmogorov Axioms — The 3 Rules That Govern All Probability

### Plain English First

Everything in probability theory follows from just **3 simple rules**.

**Axiom 1: Non-negativity** — probabilities are never negative
> P(A) ≥ 0 for every event A

**Axiom 2: Normalization** — something must happen
> P(Ω) = 1  (the probability that *some* house gets sold is 1 = 100%)

**Axiom 3: Additivity** — if events can't overlap, probabilities add up
> If A ∩ B = ∅ (events are mutually exclusive):  P(A ∪ B) = P(A) + P(B)

### Everything Else is Derived From These 3:

| Rule | Derived from | Formula |
|------|-------------|----------|
| Complement | Axioms 2+3 | P(not A) = 1 − P(A) |
| Impossible event | Axiom 2+3 | P(∅) = 0 |
| General union | Axioms 1+3 | P(A∪B) = P(A)+P(B)−P(A∩B) |
| Subset | Axiom 3 | If A⊆B then P(A) ≤ P(B) |

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Verify all Kolmogorov axioms and derived rules on the housing data
# ─────────────────────────────────────────────────────────────────

Omega = np.ones(N_HOUSES, dtype=bool)   # the entire sample space
Empty = np.zeros(N_HOUSES, dtype=bool)  # the empty event

print("VERIFYING KOLMOGOROV AXIOMS ON 100,000 HOUSE SALES")
print("=" * 60)

print("\nAXIOM 1: Non-negativity — P(A) ≥ 0")
print(f"  P(price>$400k) = {P(event_A):.4f}  ≥ 0? {P(event_A) >= 0}  ✓")
print(f"  P(price<$200k) = {P(event_B):.4f}  ≥ 0? {P(event_B) >= 0}  ✓")
print(f"  P(beds≥4)      = {P(event_C):.4f}  ≥ 0? {P(event_C) >= 0}  ✓")

print("\nAXIOM 2: Normalization — P(Ω) = 1")
print(f"  P(any house sells) = {P(Omega):.4f}  = 1? {abs(P(Omega) - 1.0) < 1e-10}  ✓")

print("\nAXIOM 3: Additivity — P(A∪B) = P(A)+P(B) when A∩B = ∅")
# A = price>400k, B = price<200k  (cannot overlap!)
A_and_B = event_A & event_B    # intersection: >400k AND <200k  (impossible)
A_or_B  = event_A | event_B    # union: >400k OR <200k
print(f"  A = {{price>$400k}},  B = {{price<$200k}}")
print(f"  P(A∩B) = {P(A_and_B):.6f}  (should be 0 — impossible to have price both >400k and <200k)")
print(f"  P(A∪B) = {P(A_or_B):.4f}")
print(f"  P(A)+P(B) = {P(event_A)+P(event_B):.4f}")
print(f"  Equal? {abs(P(A_or_B) - (P(event_A)+P(event_B))) < 1e-10}  ✓")

print()
print("DERIVED RULES:")
print(f"  Complement:     P(price≤$400k) = 1 - P(A) = {1 - P(event_A):.4f}")
print(f"  Direct check:   P(price≤$400k) = {P(~event_A):.4f}  ← same! ✓")
print()

# Inclusion-exclusion for overlapping events
A_C = event_A & event_C    # expensive AND large
AorC = event_A | event_C   # expensive OR large
print(f"  Inclusion-exclusion (overlapping events):")
print(f"    A = {{price>$400k}}, C = {{beds≥4}}")
print(f"    P(A) = {P(event_A):.4f},  P(C) = {P(event_C):.4f},  P(A∩C) = {P(A_C):.4f}")
print(f"    P(A∪C) formula: P(A)+P(C)-P(A∩C) = {P(event_A)+P(event_C)-P(A_C):.4f}")
print(f"    P(A∪C) direct:  {P(AorC):.4f}  ← same! ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Law of Large Numbers: with more data, empirical probabilities
# converge to the true underlying probability
#
# This is WHY large training datasets are better!
# ─────────────────────────────────────────────────────────────────

# True probability: P(house is expensive) ≈ P(price > $400k)
true_probability = P(event_A)   # computed over all 100k houses (our "ground truth")

# Sample increasingly large subsets and estimate the probability
sample_sizes     = np.logspace(1, 5, 50).astype(int)   # from 10 to 100,000
estimates        = []

np.random.seed(99)
for n in sample_sizes:
    # Take a random sample of n houses
    sample_indices = np.random.choice(N_HOUSES, size=n, replace=False)
    sample_prices  = prices[sample_indices]
    # Estimate probability from this sample
    est = np.mean(sample_prices > 400_000)
    estimates.append(est)

# ─── Plot convergence ─────────────────────────────────────────────
plt.figure(figsize=(12, 5))
plt.semilogx(sample_sizes, estimates, 'steelblue', linewidth=2, label='Estimated P(price>$400k)')
plt.axhline(true_probability, color='red', linestyle='--', linewidth=2,
            label=f'True probability = {true_probability:.4f}')
# Show ±1/√n confidence band (law of large numbers bound)
upper = true_probability + 1/np.sqrt(sample_sizes)
lower = true_probability - 1/np.sqrt(sample_sizes)
plt.fill_between(sample_sizes, lower, upper, alpha=0.2, color='red',
                 label='±1/√n confidence band')
plt.xlabel('Number of houses in sample (log scale)', fontsize=11)
plt.ylabel('P(house price > $400k)', fontsize=11)
plt.title('Law of Large Numbers: More Training Data → Better Probability Estimates\n'
          '(Why bigger datasets give better ML models!)', fontsize=12)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"With    10 houses: estimate might be off by ±{1/np.sqrt(10):.2f}")
print(f"With   100 houses: estimate might be off by ±{1/np.sqrt(100):.2f}")
print(f"With  1000 houses: estimate might be off by ±{1/np.sqrt(1000):.3f}")
print(f"With 10000 houses: estimate might be off by ±{1/np.sqrt(10000):.4f}")
print()
print("This is the Law of Large Numbers — the mathematical reason why more data = better ML.")

## Part 3: Probability in ML

### How the Kolmogorov Axioms Show Up in Your Model

When your neural network outputs probabilities for house price categories  
(e.g., `[P(cheap), P(medium), P(expensive)] = [0.1, 0.3, 0.6]`):

- **Axiom 1:** Each output ≥ 0 → this is why **softmax** uses `eˣ` (always positive)
- **Axiom 2:** Sum = 1 → this is why **softmax divides by the total** to normalize
- **Axiom 3:** Categories don't overlap → probabilities of disjoint categories add up

**The softmax function IS the Kolmogorov axioms implemented in code!**

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Softmax: turning raw model outputs into valid probabilities
# (satisfying all three Kolmogorov axioms)
# ─────────────────────────────────────────────────────────────────

def softmax(logits):
    """
    Convert raw scores (logits) into probabilities.

    Why exponentiation? It ensures Axiom 1 (non-negativity).
    Why divide by sum? It ensures Axiom 2 (normalization: sum = 1).

    logits : raw scores from neural network, shape (K,)
    Returns: probability distribution over K classes, shape (K,)
    """
    # Subtract max for numerical stability (same result, avoids overflow)
    logits_shifted = logits - np.max(logits)
    exp_logits     = np.exp(logits_shifted)    # make all positive (Axiom 1)
    return exp_logits / np.sum(exp_logits)      # normalize to sum=1 (Axiom 2)

# Simulate a neural network's raw output for one house
# Three classes: cheap (<$200k), medium ($200k-$500k), expensive (>$500k)
raw_scores = np.array([1.2, 3.5, 0.8])   # arbitrary "logits" from the model

probabilities = softmax(raw_scores)

class_names = ['Cheap (<$200k)', 'Medium ($200k-$500k)', 'Expensive (>$500k)']

print("Neural Network Output → Probabilities via Softmax")
print(f"Raw scores (logits):  {raw_scores}")
print(f"Probabilities:        {probabilities.round(4)}")
print()
print("Verifying Kolmogorov Axioms on softmax output:")
print(f"  Axiom 1 (all ≥ 0):  {all(p >= 0 for p in probabilities)}  ✓")
print(f"  Axiom 2 (sum = 1):  {abs(sum(probabilities) - 1.0) < 1e-10}  (sum = {sum(probabilities):.10f})  ✓")
print(f"  Axiom 3 (additive): P(cheap or medium) = {probabilities[0]+probabilities[1]:.4f}")
print(f"                    = P(cheap) + P(medium) = {probabilities[0]:.4f} + {probabilities[1]:.4f}  ✓")
print()

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(class_names, raw_scores, color='steelblue', alpha=0.8)
axes[0].set_title('Raw Scores (Logits)\n(arbitrary, can be any value)', fontsize=11)
axes[0].set_ylabel('Score')
axes[0].axhline(0, color='black', linewidth=0.5)

colors_bar = ['green', 'orange', 'red']
axes[1].bar(class_names, probabilities, color=colors_bar, alpha=0.8)
for i, (name, prob) in enumerate(zip(class_names, probabilities)):
    axes[1].text(i, prob + 0.01, f'{prob:.3f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title(f'Probabilities after Softmax\n(all ≥ 0, sum = {probabilities.sum():.4f} = 1)', fontsize=11)
axes[1].set_ylabel('Probability')
axes[1].set_ylim(0, 1)

plt.suptitle('Softmax: Converting Raw Scores to Valid Probabilities (Kolmogorov Axioms)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## ✅ Self-Check Questions

**1.** Your neural network for house classification outputs `[0.3, -0.1, 0.8]` after softmax.  
   Is this valid? Which Kolmogorov axiom does it violate?

**2.** In your house dataset, P(bedrooms=3) = 0.4 and P(bedrooms=4) = 0.2.  
   What is P(bedrooms=3 OR bedrooms=4)? Which axiom did you use?

**3.** You train your model on 100 houses. P(expensive) in your sample is 0.28.  
   The true P(expensive) is 0.30. Is this reasonable? What explains the gap?

**4.** Can two events have P(A) = 0.7 and P(B) = 0.6 with P(A ∪ B) = 1.0?  
   Use the inclusion-exclusion formula to check.

**5.** Your model predicts house price class. The output probabilities sum to 0.95.  
   Which axiom is violated, and what went wrong technically?

<details>
<summary>Click to see answers</summary>

1. **Invalid!** Axiom 1 (non-negativity) is violated: -0.1 < 0. A valid probability distribution must have all values ≥ 0. This would mean softmax was not applied, or there's a bug.

2. P(3 OR 4 bedrooms) = P(3) + P(4) = 0.4 + 0.2 = **0.6**. Used **Axiom 3** (additivity for mutually exclusive events — a house can't have both 3 and 4 bedrooms).

3. **Yes, very reasonable.** The gap of 0.02 is within the expected sampling error of ±1/√100 = ±0.10 for 100 samples. The Law of Large Numbers says estimates converge to the true value with more data.

4. Yes! P(A∪B) = P(A)+P(B)-P(A∩B) → 1.0 = 0.7+0.6-P(A∩B) → P(A∩B) = 0.3. So 30% of houses are both expensive AND large, which is perfectly valid.

5. **Axiom 2** (normalization: sum must = 1). Technically, the softmax computation had a numerical error or the logits overflowed. The fix is to subtract the max logit before computing exp (numerical stability trick).

</details>

---

## 📌 Summary

| Concept | Definition | House price example |
|---------|-----------|--------------------|
| **Sample space Ω** | All possible outcomes | All possible house prices |
| **Event** | A subset of Ω | {price > $400k} |
| **P(A)** | Fraction of outcomes in A | 23% of houses sell above $400k |
| **Axiom 1** | P(A) ≥ 0 | Probabilities are never negative |
| **Axiom 2** | P(Ω) = 1 | Every house has some price |
| **Axiom 3** | P(A∪B)=P(A)+P(B) when A∩B=∅ | Mutually exclusive outcomes add up |
| **Softmax** | Enforces all 3 axioms | Converts neural network outputs to valid probabilities |

**➡️ Next Notebook:** Conditional Probability & Independence — what happens to probabilities when you have extra information.